# 07. Exportar CSV da aplicação Splink

Gera dois CSV a partir das bases V3 do Luis (`censo_limpo` / `cpf_limpo`),
com as colunas na ordem pedida, todas como string.

Corte: remove da censo e da CPF todo `id_morador` / `cpf` que aparece
em `LISTA_OURO_AVALIACAO_ARQUIVO`. Conta antes e depois; exporta só o
recorte já excluído.

Mapeamento Luis → CSV: `nome_fonetico` → `nome_completo_phon`,
`fon_primeiro` / `fon_ultimo`, `nome_fonetico_mae` → `nome_mae_phon`,
`cod_mun` → `cod_municipio`, `logradouro` → `logradouro_norm`.

`id_seq` numera 1..n. CSV **sem cabeçalho**: `id_seq` na primeira coluna.

Saída em `OUTPUT_DIR`:
- `censo_aplicacao_splink.csv`
- `cpf_aplicacao_splink.csv`


In [ ]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    LISTA_OURO_AVALIACAO_ARQUIVO,
    OUTPUT_DIR,
    cpf_norm_sql,
    ensure_output_dir,
    get_connection,
    lista_ouro_colunas,
    print_paths,
    require_input,
)

CENSO_V3_LUIS = Path(
    '/data/singed/capefe/scripts_luis/Pareamento Probabilístico/'
    'V3 20260925/censo_limpo.parquet'
)
CPF_V3_LUIS = Path(
    '/data/singed/capefe/scripts_luis/Pareamento Probabilístico/'
    'V3 20260925/cpf_limpo.parquet'
)

print_paths()
ensure_output_dir()
require_input(CENSO_V3_LUIS, label='CENSO_V3_LUIS')
require_input(CPF_V3_LUIS, label='CPF_V3_LUIS')
require_input(LISTA_OURO_AVALIACAO_ARQUIVO, label='LISTA_OURO_AVALIACAO')

con = get_connection()

censo_path = str(CENSO_V3_LUIS).replace("'", "''")
cpf_path = str(CPF_V3_LUIS).replace("'", "''")
ouro_path = str(LISTA_OURO_AVALIACAO_ARQUIVO).replace("'", "''")

ouro_cols = {
    r[0] for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{ouro_path}') LIMIT 0"
    ).fetchall()
}
id_ouro, cpf_ouro = lista_ouro_colunas(ouro_cols)

con.execute(f'''
CREATE OR REPLACE TABLE lista_ouro AS
SELECT
    CAST({id_ouro} AS VARCHAR) AS id_censo,
    {cpf_norm_sql(cpf_ouro)} AS cpf_norm
FROM read_parquet('{ouro_path}')
WHERE {id_ouro} IS NOT NULL OR {cpf_ouro} IS NOT NULL
''')

display(con.execute('''
SELECT
    COUNT(*) AS n_pares,
    COUNT(DISTINCT id_censo) AS n_id_censo,
    COUNT(DISTINCT cpf_norm) AS n_cpf
FROM lista_ouro
''').df())

n_censo_antes = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{censo_path}')"
).fetchone()[0]
n_cpf_antes = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{cpf_path}')"
).fetchone()[0]

# Censo: fora se id_morador ou cpf está na ouro.
con.execute(f'''
CREATE OR REPLACE TABLE censo_export AS
SELECT *
FROM read_parquet('{censo_path}') a
WHERE CAST(a.id_morador AS VARCHAR) NOT IN (
    SELECT id_censo FROM lista_ouro WHERE id_censo IS NOT NULL
)
  AND (
    {cpf_norm_sql('a.cpf')} IS NULL
    OR {cpf_norm_sql('a.cpf')} NOT IN (
        SELECT cpf_norm FROM lista_ouro WHERE cpf_norm IS NOT NULL
    )
)
''')

# CPF: fora se cpf está na ouro.
con.execute(f'''
CREATE OR REPLACE TABLE cpf_export AS
SELECT *
FROM read_parquet('{cpf_path}') a
WHERE {cpf_norm_sql('a.cpf')} IS NULL
   OR {cpf_norm_sql('a.cpf')} NOT IN (
       SELECT cpf_norm FROM lista_ouro WHERE cpf_norm IS NOT NULL
   )
''')

n_censo_depois = con.execute('SELECT COUNT(*) FROM censo_export').fetchone()[0]
n_cpf_depois = con.execute('SELECT COUNT(*) FROM cpf_export').fetchone()[0]

print(f'Censo: {n_censo_antes:,} → {n_censo_depois:,}'
      f' (excluídos {n_censo_antes - n_censo_depois:,})')
print(f'CPF:   {n_cpf_antes:,} → {n_cpf_depois:,}'
      f' (excluídos {n_cpf_antes - n_cpf_depois:,})')

COLS = [
    'id_seq',
    'nome_completo_phon',
    'primeiro_nome_phon',
    'ultimo_nome_phon',
    'data_nascimento',
    'ano_nascimento',
    'mes_nascimento',
    'dia_nascimento',
    'idade',
    'nome_mae_phon',
    'logradouro_norm',
    'cep',
    'cod_municipio',
    'sexo',
]

select_cols = '''
    CAST(ROW_NUMBER() OVER (ORDER BY unique_id) AS VARCHAR) AS id_seq,
    CAST(nome_fonetico AS VARCHAR) AS nome_completo_phon,
    CAST(fon_primeiro AS VARCHAR) AS primeiro_nome_phon,
    CAST(fon_ultimo AS VARCHAR) AS ultimo_nome_phon,
    CAST(data_nascimento AS VARCHAR) AS data_nascimento,
    CAST(
        CASE
            WHEN data_nascimento IS NULL
              OR TRIM(CAST(data_nascimento AS VARCHAR)) = '' THEN NULL
            ELSE substr(CAST(data_nascimento AS VARCHAR), 1, 4)
        END AS VARCHAR
    ) AS ano_nascimento,
    CAST(
        CASE
            WHEN data_nascimento IS NULL
              OR TRIM(CAST(data_nascimento AS VARCHAR)) = '' THEN NULL
            ELSE substr(CAST(data_nascimento AS VARCHAR), 6, 2)
        END AS VARCHAR
    ) AS mes_nascimento,
    CAST(
        CASE
            WHEN data_nascimento IS NULL
              OR TRIM(CAST(data_nascimento AS VARCHAR)) = '' THEN NULL
            ELSE substr(CAST(data_nascimento AS VARCHAR), 9, 2)
        END AS VARCHAR
    ) AS dia_nascimento,
    CAST(idade AS VARCHAR) AS idade,
    CAST(nome_fonetico_mae AS VARCHAR) AS nome_mae_phon,
    CAST(logradouro AS VARCHAR) AS logradouro_norm,
    CAST(cep AS VARCHAR) AS cep,
    CAST(cod_mun AS VARCHAR) AS cod_municipio,
    CAST(sexo AS VARCHAR) AS sexo
'''

out_censo = OUTPUT_DIR / 'censo_aplicacao_splink.csv'
out_cpf = OUTPUT_DIR / 'cpf_aplicacao_splink.csv'

con.execute(f'''
COPY (
    SELECT {select_cols}
    FROM censo_export
) TO '{out_censo}' (HEADER false, DELIMITER ',')
''')

con.execute(f'''
COPY (
    SELECT {select_cols}
    FROM cpf_export
) TO '{out_cpf}' (HEADER false, DELIMITER ',')
''')

print('Colunas:', COLS)
print('Censo CSV:', out_censo, '| linhas:', f'{n_censo_depois:,}')
print('CPF CSV:  ', out_cpf, '| linhas:', f'{n_cpf_depois:,}')

cols_csv = ', '.join(f"'{c}': 'VARCHAR'" for c in COLS)
display(con.execute(f'''
SELECT * FROM read_csv(
    '{out_censo}',
    header=false,
    columns={{{cols_csv}}}
) LIMIT 3
''').df())
display(con.execute(f'''
SELECT * FROM read_csv(
    '{out_cpf}',
    header=false,
    columns={{{cols_csv}}}
) LIMIT 3
''').df())

con.close()
